In [ ]:
# from google.colab import drive
# drive.mount("/content/drive", force_remount=True) # Colab Specific

In [ ]:
!pip -q install torch==2.9.0+cu128 torchvision --extra-index-url https://download.pytorch.org/whl/cu128
!pip -q install transformers==5.1.0 datasets==4.5.0 scikit-learn==1.8.0 accelerate sentence-transformers evaluate

In [ ]:
import os, json, time, random
from pathlib import Path

In [ ]:
PROJECT_ROOT = "."

DATA_DIR      = f"{PROJECT_ROOT}/data"
KB_DIR        = f"{DATA_DIR}/knowledge_base"
KB_FILE       = f"{KB_DIR}/knowledge_base_v2.jsonl"

RETR_ROOT     = f"{PROJECT_ROOT}/retrieval_cache"
OUT_ROOT      = f"{PROJECT_ROOT}/outputs/retrieval"

Path(RETR_ROOT).mkdir(parents=True, exist_ok=True)
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("KB_FILE:", KB_FILE)
print("RETR_ROOT:", RETR_ROOT)
print("OUT_ROOT:", OUT_ROOT)

In [ ]:
import torch, transformers, datasets, sklearn
import numpy as np
import pandas as pd

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("sklearn:", sklearn.__version__)

In [ ]:
from datetime import datetime
import os

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MODEL_NAME = "roberta-base"
RETRIEVER_NAME = "Qwen/Qwen3-Embedding-0.6B"

MAX_LEN = 384
TRAIN_BS = 16
EVAL_BS = 32
LR = 2e-5
EPOCHS = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOP_PATIENCE = 2
AGG_METHOD = "mean"

TOP_K = 3
MIN_SCORE = 0.4
USE_CACHE = False

QUERY_PROMPT = (
    "Instruct: Consider the full patient interview chunk. Retrieve depression symptom knowledge "
    "only when the chunk as a whole points to depressive symptoms, either explicitly, implicitly, "
    "or through tone and mood. Do not retrieve for unrelated or merely negative content.\n"
    "Query: "
)

RUN_NAME = f"retrieval_{MODEL_NAME.replace('/','_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_top_k_{TOP_K}_min_score_{str(MIN_SCORE).replace(".", "")}_retriever_{RETRIEVER_NAME.lower().split("/")[-1]}"

RUN_DIR = f"{OUT_ROOT}/{RUN_NAME}"
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

config = dict(
    run_name=RUN_NAME,
    model_name=MODEL_NAME,
    retriever_name=RETRIEVER_NAME,
    max_len=MAX_LEN,
    train_bs=TRAIN_BS,
    eval_bs=EVAL_BS,
    lr=LR,
    epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    seed=SEED,
    early_stop_patience=EARLY_STOP_PATIENCE,
    agg_method=AGG_METHOD,
    top_k=TOP_K,
    min_score=MIN_SCORE,
    kb_file=KB_FILE,
    data_files=dict(
        train=f"{DATA_DIR}/train_cased.jsonl",
        validation=f"{DATA_DIR}/dev_cased.jsonl",
        test=f"{DATA_DIR}/test_cased.jsonl",
    )
)

with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("RUN_DIR:", RUN_DIR)
print(json.dumps(config, indent=2)[:1000], "...")

In [ ]:
from datasets import load_dataset

data_files = config["data_files"]
raw = load_dataset("json", data_files=data_files)

required = {"pid", "chunk_id", "text", "label"}
for split in raw.keys():
    cols = set(raw[split].column_names)
    missing = required - cols
    if missing:
        raise ValueError(f"Split '{split}' missing columns: {missing}")

def fix_types(ex):
    ex["pid"] = int(ex["pid"])
    ex["chunk_id"] = int(ex["chunk_id"])
    ex["label"] = int(ex["label"])
    return ex

raw = raw.map(fix_types)

def summarize(split):
    labels = np.array(raw[split]["label"])
    pids = np.array(raw[split]["pid"])
    uniq_pids = np.unique(pids)

    pid_to_label = {}
    for pid, lab in zip(pids, labels):
        pid_to_label.setdefault(pid, lab)
    pat_labels = np.array(list(pid_to_label.values()))

    return {
        "n_rows": len(labels),
        "n_patients": len(uniq_pids),
        "chunk_depressed_frac": float((labels == 1).mean()),
        "patient_depressed_frac": float((pat_labels == 1).mean()),
    }

stats = {s: summarize(s) for s in ["train", "validation", "test"]}
print(json.dumps(stats, indent=2))

with open(f"{RUN_DIR}/data_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

In [ ]:
kb_entries = []
with open(KB_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            kb_entries.append(json.loads(line))

print("KB entries:", len(kb_entries))
print("Sample entry:", kb_entries[0])

required_kb = {"kb_id", "category", "title", "text"}
for i, ex in enumerate(kb_entries):
    missing = required_kb - set(ex.keys())
    if missing:
        raise ValueError(f"KB entry {i} missing fields: {missing}")

In [ ]:
def kb_to_embed_text(ex):
    return f"{ex['category']}. {ex['title']}. {ex['text']}"

kb_ids = [ex["kb_id"] for ex in kb_entries]
kb_embed_texts = [kb_to_embed_text(ex) for ex in kb_entries]
kb_raw_texts = [ex["text"] for ex in kb_entries]

print(kb_embed_texts[0])

In [ ]:
from sentence_transformers import SentenceTransformer

# kb_emb_path = f"{RETR_ROOT}/kb_embeddings.npy"
# kb_meta_path = f"{RETR_ROOT}/kb_meta.json"
retr_tag = RETRIEVER_NAME.split("/")[-1]
kb_tag = Path(KB_FILE).stem

kb_emb_path = f"{RETR_ROOT}/kb_embeddings_{kb_tag}_{retr_tag}.npy"
kb_meta_path = f"{RETR_ROOT}/kb_meta_{kb_tag}_{retr_tag}.json"

retriever = SentenceTransformer(
    RETRIEVER_NAME,
    tokenizer_kwargs={"padding_side": "left"},
)

if USE_CACHE and os.path.exists(kb_emb_path) and os.path.exists(kb_meta_path):
    kb_embeddings = np.load(kb_emb_path)
    with open(kb_meta_path, "r", encoding="utf-8") as f:
        kb_meta = json.load(f)
    print("Loaded cached KB embeddings:", kb_embeddings.shape)
else:
    kb_embeddings = retriever.encode(
        kb_embed_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    np.save(kb_emb_path, kb_embeddings)

    kb_meta = {
        "kb_ids": kb_ids,
        "kb_embed_texts": kb_embed_texts,
        "kb_raw_texts": kb_raw_texts,
        "kb_entries": kb_entries,
        "retriever_name": RETRIEVER_NAME
    }

    with open(kb_meta_path, "w", encoding="utf-8") as f:
        json.dump(kb_meta, f, indent=2, ensure_ascii=False)

    print("Saved KB embeddings:", kb_embeddings.shape)

In [ ]:
def retrieve_topk_thresholded(query_embedding, kb_embeddings, k=3, min_score=0.30):
    scores = np.dot(kb_embeddings, query_embedding)
    top_idx = np.argsort(scores)[::-1][:k]

    kept_idx = []
    kept_scores = []
    for idx in top_idx:
        s = float(scores[idx])
        if s >= min_score:
            kept_idx.append(idx)
            kept_scores.append(s)

    return kept_idx, kept_scores

def build_radc_text(chunk_text, retrieved_entries):
    if len(retrieved_entries) == 0:
        return f"patient: {chunk_text}"

    context = " </s> ".join([x["text"] for x in retrieved_entries])
    return f"patient: {chunk_text} </s></s> context: {context}"

In [ ]:
def add_retrieval_to_split(ds, split_name, top_k=3, min_score=0.30, batch_size=128):
    texts = ds["text"]

    query_embeddings = retriever.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        prompt=QUERY_PROMPT,
)

    retrieved_kb_ids = []
    retrieved_kb_texts = []
    retrieval_scores = []
    radc_texts = []
    n_empty = 0

    for i in range(len(texts)):
        idxs, scores = retrieve_topk_thresholded(
            query_embeddings[i],
            kb_embeddings,
            k=top_k,
            min_score=min_score
        )

        entries = [kb_entries[j] for j in idxs]

        if len(entries) == 0:
            n_empty += 1

        retrieved_kb_ids.append([x["kb_id"] for x in entries])
        retrieved_kb_texts.append([x["text"] for x in entries])
        retrieval_scores.append([float(s) for s in scores])
        radc_texts.append(build_radc_text(texts[i], entries))

    ds = ds.add_column("retrieved_kb_ids", retrieved_kb_ids)
    ds = ds.add_column("retrieved_kb_texts", retrieved_kb_texts)
    ds = ds.add_column("retrieval_scores", retrieval_scores)
    ds = ds.add_column("radc_text", radc_texts)

    print(f"{split_name}: added retrieval fields")
    print(f"{split_name}: empty retrievals = {n_empty}/{len(texts)} ({100*n_empty/len(texts):.2f}%)")

    return ds

In [ ]:
# train_cache = f"{RETR_ROOT}/train_with_retrieval.jsonl"
# dev_cache   = f"{RETR_ROOT}/dev_with_retrieval.jsonl"
# test_cache  = f"{RETR_ROOT}/test_with_retrieval.jsonl"

retr_tag = RETRIEVER_NAME.split("/")[-1]
cache_tag = f"{Path(KB_FILE).stem}_{retr_tag}_top{TOP_K}_min{str(MIN_SCORE).replace('.','p')}"

train_cache = f"{RETR_ROOT}/train_with_retrieval_{cache_tag}.jsonl"
dev_cache   = f"{RETR_ROOT}/dev_with_retrieval_{cache_tag}.jsonl"
test_cache  = f"{RETR_ROOT}/test_with_retrieval_{cache_tag}.jsonl"

def save_jsonl_from_dataset(ds, path):
    with open(path, "w", encoding="utf-8") as f:
        for ex in ds:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

if USE_CACHE and all(os.path.exists(x) for x in [train_cache, dev_cache, test_cache]):
    aug = load_dataset("json", data_files={
        "train": train_cache,
        "validation": dev_cache,
        "test": test_cache
    })
    print("Loaded cached retrieval-augmented datasets")
else:
    aug = {}
    aug["train"] = add_retrieval_to_split(raw["train"], "train", top_k=TOP_K, min_score=MIN_SCORE)
    aug["validation"] = add_retrieval_to_split(raw["validation"], "validation", top_k=TOP_K, min_score=MIN_SCORE)
    aug["test"] = add_retrieval_to_split(raw["test"], "test", top_k=TOP_K, min_score=MIN_SCORE)

    save_jsonl_from_dataset(aug["train"], train_cache)
    save_jsonl_from_dataset(aug["validation"], dev_cache)
    save_jsonl_from_dataset(aug["test"], test_cache)

    from datasets import DatasetDict
    aug = DatasetDict(aug)

    print("Saved retrieval-augmented datasets")

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
collator = DataCollatorWithPadding(tokenizer)

def tok(batch):
    return tokenizer(batch["radc_text"], truncation=True, max_length=MAX_LEN)

tokenized = aug.map(tok, batched=True)

tokenized.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label", "pid", "chunk_id"]
)

print(tokenized["train"][0])

In [ ]:
lengths = [len(tokenizer(x)["input_ids"]) for x in aug["train"]["radc_text"][:1000]]
print("Token length stats on first 1000 train examples:")
print("mean:", np.mean(lengths))
print("p90 :", np.percentile(lengths, 90))
print("p95 :", np.percentile(lengths, 95))
print("max :", np.max(lengths))

In [ ]:
import evaluate
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

acc = evaluate.load("accuracy")

def chunk_metrics_from_probs(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    out = {
        "accuracy": float(acc.compute(predictions=preds, references=labels)["accuracy"]),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "auroc": float(roc_auc_score(labels, probs)) if len(np.unique(labels)) > 1 else float("nan"),
        "threshold": float(threshold),
    }
    return out

def aggregate_patient(probs, labels, pids, method="mean"):
    df = pd.DataFrame({"pid": pids, "prob": probs, "label": labels})
    if method == "mean":
        agg = df.groupby("pid", as_index=False).agg(prob=("prob", "mean"), label=("label", "max"))
    elif method == "max":
        agg = df.groupby("pid", as_index=False).agg(prob=("prob", "max"), label=("label", "max"))
    else:
        raise ValueError("method must be 'mean' or 'max'")
    return agg

def patient_metrics_from_probs(probs, labels, pids, method="mean", threshold=0.5):
    agg = aggregate_patient(probs, labels, pids, method=method)
    return chunk_metrics_from_probs(agg["prob"].values, agg["label"].values, threshold=threshold), agg

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return chunk_metrics_from_probs(probs, labels_np, threshold=0.5)

args = TrainingArguments(
    output_dir=f"{RUN_DIR}/checkpoints",
    eval_strategy="epoch",
    # save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="f1",
    greater_is_better=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=False,
    logging_steps=50,
    data_seed=SEED,
    seed=SEED,
    full_determinism=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)]
)

In [ ]:
train_out = trainer.train()
# trainer.save_model(f"{RUN_DIR}/best_model")
# print("Saved best model to:", f"{RUN_DIR}/best_model")

In [ ]:
dev_chunk = trainer.evaluate(tokenized["validation"])
test_chunk = trainer.evaluate(tokenized["test"])

print("DEV chunk metrics:", dev_chunk)
print("TEST chunk metrics:", test_chunk)

with open(f"{RUN_DIR}/chunk_metrics.json", "w") as f:
    json.dump({"dev": dev_chunk, "test": test_chunk}, f, indent=2)

In [ ]:
def predict_split(split_name):
    pred = trainer.predict(tokenized[split_name])
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    labels_np = np.array(aug[split_name]["label"])
    pids = np.array(aug[split_name]["pid"])
    chunk_ids = np.array(aug[split_name]["chunk_id"])

    df = pd.DataFrame({
        "pid": pids,
        "chunk_id": chunk_ids,
        "prob_depressed": probs,
        "label": labels_np
    })
    return df

dev_df = predict_split("validation")
test_df = predict_split("test")

dev_df.to_csv(f"{RUN_DIR}/dev_chunk_preds.csv", index=False)
test_df.to_csv(f"{RUN_DIR}/test_chunk_preds.csv", index=False)

print(dev_df.head())

In [ ]:
_, dev_pat_df = patient_metrics_from_probs(
    probs=dev_df["prob_depressed"].values,
    labels=dev_df["label"].values,
    pids=dev_df["pid"].values,
    method=AGG_METHOD,
    threshold=0.5
)

dev_probs = dev_pat_df["prob"].values
dev_labels = dev_pat_df["label"].values

best_f1 = -1
best_threshold = 0.5

for t in np.linspace(0.05, 0.95, 181):
    preds = (dev_probs >= t).astype(int)
    f1 = f1_score(dev_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(t)

print("Best threshold from DEV:", best_threshold)
print("Best DEV F1:", best_f1)

dev_pat_metrics, dev_pat_df = patient_metrics_from_probs(
    probs=dev_df["prob_depressed"].values,
    labels=dev_df["label"].values,
    pids=dev_df["pid"].values,
    method=AGG_METHOD,
    threshold=best_threshold
)

test_pat_metrics, test_pat_df = patient_metrics_from_probs(
    probs=test_df["prob_depressed"].values,
    labels=test_df["label"].values,
    pids=test_df["pid"].values,
    method=AGG_METHOD,
    threshold=best_threshold
)

print("DEV patient metrics (tuned):", dev_pat_metrics)
print("TEST patient metrics (tuned):", test_pat_metrics)

dev_pat_df.to_csv(f"{RUN_DIR}/dev_patient_preds.csv", index=False)
test_pat_df.to_csv(f"{RUN_DIR}/test_patient_preds.csv", index=False)

with open(f"{RUN_DIR}/patient_metrics.json", "w") as f:
    json.dump(
        {"threshold": best_threshold, "dev": dev_pat_metrics, "test": test_pat_metrics},
        f,
        indent=2
    )

In [ ]:
summary = {
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "retriever_name": RETRIEVER_NAME,
    "data_stats": stats,
    "chunk_metrics": {"dev": dev_chunk, "test": test_chunk},
    "patient_metrics": {"dev": dev_pat_metrics, "test": test_pat_metrics},
    "agg_method": AGG_METHOD,
    "threshold": best_threshold,
    "dev_best_f1": best_f1,
    "top_k": TOP_K,
    "max_len": MAX_LEN,
    "notes": "Retrieval-augmented training. Input format: patient chunk + top-k retrieved KB entries."
}

with open(f"{RUN_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote:", f"{RUN_DIR}/summary.json")
print(json.dumps(summary, indent=2)[:1400], "...")